<a href="https://colab.research.google.com/github/baynobu/pembelajaran-mesin-26/blob/main/JS15_Praktikum_dan_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jobsheet 15 - Pipeline Machine Learning & Deployment
**Mata Kuliah:** Pembelajaran Mesin / Machine Learning
**Modul:** JS15 - Siklus Lengkap Machine Learning Pipeline & Deployment Cloud (Hugging Face Spaces)

## 0. Konfigurasi Lingkungan (Google Colab / Lokal)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

---
# Praktikum 1: Klasifikasi Gambar (Day vs Night) dengan Neural Network & Ekspor Artefak Model

### Langkah 1 - Import Library

In [ ]:
from pathlib import Path
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import cv2

from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models

### Langkah 2 - Load Dataset

In [ ]:
def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for folder in p.glob('*'):
        if not folder.is_dir():
            continue
        label = folder.name
        for file in folder.glob('*.jpg'):
            img = cv2.imread(str(file))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_list.append((img, label))
    return img_list

if os.path.exists("images/training"):
    train_dir = "images/training"
    test_dir  = "images/test"
elif os.path.exists("JS15/images/training"):
    train_dir = "JS15/images/training"
    test_dir  = "JS15/images/test"
else:
    train_dir = "/content/drive/MyDrive/images/training/"
    test_dir  = "/content/drive/MyDrive/images/test/"

train_img = load_dataset(train_dir)
test_img  = load_dataset(test_dir)

print(f"Jumlah data training: {len(train_img)}")
print(f"Jumlah data testing : {len(test_img)}")

### Langkah 3 - Preprocessing (Resize & Label Encoding)

In [ ]:
def resize_image(img, size=(256, 256)):
    return cv2.resize(img, size)

def label_encoder(label):
    return 1 if label.lower() == 'day' else 0

def preprocess(img_list):
    X = []
    y = []
    for img, label in img_list:
        img_std = resize_image(img)
        X.append(img_std)
        y.append(label_encoder(label))
    return X, y

X_train_img, y_train = preprocess(train_img)
X_test_img,  y_test  = preprocess(test_img)

### Langkah 4 - Ekstraksi Fitur HOG (Histogram of Oriented Gradients)

In [ ]:
def extract_hog(X_imgs):
    feats = []
    for img in X_imgs:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        hog_feat = hog(gray,
                       orientations=9,
                       pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2),
                       block_norm='L2-Hys',
                       visualize=False,
                       feature_vector=True)
        feats.append(hog_feat)
    return np.array(feats)

X_train_feat = extract_hog(X_train_img)
X_test_feat  = extract_hog(X_test_img)
print(f"Dimensi Vektor Fitur HOG: {X_train_feat.shape}")

### Langkah 5 - Normalisasi Fitur

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled  = scaler.transform(X_test_feat)

### Langkah 6 - Split Data Training & Validation

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.2,
    random_state=42
)

### Langkah 7 - Konversi Label ke Numpy Array

In [ ]:
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

### Langkah 8 - Membangun Arsitektur Model Neural Network

In [ ]:
input_dim = X_train.shape[1]

model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Langkah 9 - Training Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val)
)

### Langkah 10 - Evaluasi Model

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test)
print(f"Akurasi Pengujian Test: {test_acc:.4f} ({test_acc*100:.2f}%)")

### Langkah 11 - Prediksi, Laporan Klasifikasi, Confusion Matrix & Visualisasi History

In [ ]:
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Night (0)', 'Day (1)']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

### Langkah 12 - Menyimpan Model dan Scaler

In [ ]:
model.save('day_night_model.h5')
print("Model berhasil disimpan: day_night_model.h5")

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler berhasil disimpan: scaler.pkl")

os.makedirs('daynight-project', exist_ok=True)
model.save('daynight-project/day_night_model.h5')
with open('daynight-project/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Artefak model dan scaler juga disalin ke folder daynight-project/")

---
# Praktikum 2: Deployment ke Hugging Face Spaces (Docker + Flask)

### Langkah 1 s.d. 4 - Pembuatan Server `app.py`, `requirements.txt`, dan `Dockerfile`
File-file deployment telah dibuat dan siap di dalam direktori `daynight-project/`.

In [ ]:
with open('daynight-project/app.py', 'r', encoding='utf-8') as f:
    print("--- Konten app.py ---")
    print(f.read()[:500] + "\n...")

In [ ]:
with open('daynight-project/requirements.txt', 'r', encoding='utf-8') as f:
    print("--- Konten requirements.txt ---")
    print(f.read())

In [ ]:
with open('daynight-project/Dockerfile', 'r', encoding='utf-8') as f:
    print("--- Konten Dockerfile ---")
    print(f.read())

### Langkah 5 & 6 - Langkah Deploy ke Hugging Face Spaces melalui Git CLI
1. Buat **New Space** di Hugging Face (SDK: Docker, License: Apache 2.0 / MIT, Visibility: Public).
2. Buat Access Token tipe **Write** di menu *Settings > Access Tokens*.
3. Buka Terminal / Git Bash lalu clone repository space:
```bash
git clone https://huggingface.co/spaces/USERNAME/SPACE_NAME
```
4. Salin seluruh isi folder `daynight-project/` (`app.py`, `Dockerfile`, `requirements.txt`, `README.md`, `day_night_model.h5`, `scaler.pkl`) ke dalam direktori hasil clone.
5. Jalankan perintah push:
```bash
cd SPACE_NAME
git add .
git commit -m "Deploy Day vs Night Classifier with Flask and Docker"
git push
```
6. Hugging Face Spaces akan otomatis melakukan proses build image Docker dan aplikasi web live pada port 7860.

---
# Tugas: Menerapkan Pipeline Deployment untuk Model Project Machine Learning
Penerapan pipeline MLOps yang sama untuk model proyek mandiri (misal klasifikasi MNIST atau model proyek tim).

### 1. Struktur Standar Proyek Deployment

In [ ]:
project_deploy_structure = {
    "project-deploy/": [
        "app.py              # Web server Flask untuk memproses input & inferensi model",
        "requirements.txt    # Daftar dependensi library Python",
        "Dockerfile          # Kontainerisasi lingkungan runtime",
        "README.md           # Metadata Hugging Face Space",
        "project_model.h5    # File bobot model hasil training",
        "project_scaler.pkl  # Objek pra-pemrosesan (jika digunakan)"
    ]
}

for root, files in project_deploy_structure.items():
    print(root)
    for f in files:
        print(f"  └── {f}")

### 2. Template Server Flask untuk Model Project (`app_project.py`)

In [ ]:
app_project_code = """import numpy as np
import tensorflow as tf
from flask import Flask, request, jsonify, render_template_string

app = Flask(__name__)

model = tf.keras.models.load_model('project_model.h5')

@app.route('/', methods=['GET'])
def index():
    return render_template_string('<h1>Project Model Inference API</h1><p>Endpoint POST /predict siap menerima data inferensi.</p>')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json(force=True)
        inputs = np.array(data['inputs'])
        preds = model.predict(inputs)
        return jsonify({'predictions': preds.tolist()})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=7860)
"""

with open('daynight-project/app_project_template.py', 'w', encoding='utf-8') as f:
    f.write(app_project_code)

print("Template app_project_template.py berhasil digenerate.")

### 3. Prosedur Eksekusi Deployment Proyek
1. Pastikan model dengan akurasi terbaik diekspor dalam format `.h5` atau format yang kompatibel.
2. Siapkan preprocessor (`scaler.pkl` atau tokenizers/encoders) yang konsisten dengan tahap pelatihan.
3. Buat Space baru bertipe Docker di Hugging Face.
4. Clone repo space, pindahkan file aplikasi dan artefak model, kemudian lakukan `git add`, `git commit`, dan `git push`.